In [1]:
import os
import sys
import subprocess

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION"] = "2"


def _ensure_transformers_importable():
    def _purge_protobuf_modules():
        for k in list(sys.modules.keys()):
            if k.startswith("google.protobuf") or k == "google.protobuf":
                del sys.modules[k]

    try:
        import google.protobuf  # noqa: F401
        from google.protobuf import __version__ as pb_ver  # noqa: F401
    except Exception:
        pb_ver = None

    need_install = True
    if pb_ver is not None:
        try:
            major = int(str(pb_ver).split(".")[0])
            need_install = not (major < 5)
        except Exception:
            need_install = True

    if need_install:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "protobuf<5"]
        )
        _purge_protobuf_modules()

    try:
        import transformers  # noqa: F401

        return
    except Exception as e:
        msg = repr(e)
        if (
            ("protobuf" in msg.lower())
            or ("MessageFactory" in msg)
            or ("GetPrototype" in msg)
        ):
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", "protobuf<5"]
            )
            _purge_protobuf_modules()
            import importlib

            importlib.invalidate_caches()
            import transformers  # noqa: F401

            return
        raise


_ensure_transformers_importable()

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from transformers import BertTokenizerFast, BertModel


class config:
    MAX_LEN = 141
    TRAIN_BATCH_SIZE = 40
    VALID_BATCH_SIZE = 16
    EPOCHS = 10

    BERT_PATH = "bert-base-uncased"
    MODEL_PATH = "model.bin"
    TRAINING_FILE = "/kaggle/input/tweet-sentiment-extraction/train.csv"

    TOKENIZER = None


def _init_tokenizer_offline():
    candidates = [
        ("/kaggle/input/bert-base-uncased/bert-base-uncased", True),
        ("/kaggle/input/bert-base-uncased", True),
        ("/kaggle/input/bertbaseuncased", True),
        (config.BERT_PATH, True),
        (config.BERT_PATH, False),
    ]
    last_err = None
    for path, local_only in candidates:
        try:
            tok = BertTokenizerFast.from_pretrained(path, local_files_only=local_only)
            return tok
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(
        f"Could not initialize tokenizer. Tried: {[c[0] for c in candidates]}. "
        f"Last error: {repr(last_err)}"
    )


config.TOKENIZER = _init_tokenizer_offline()


class BERTBaseUncased(nn.Module):
    def __init__(self):
        super(BERTBaseUncased, self).__init__()
        model_candidates = [
            ("/kaggle/input/bert-base-uncased/bert-base-uncased", True),
            ("/kaggle/input/bert-base-uncased", True),
            (config.BERT_PATH, True),
            (config.BERT_PATH, False),
        ]
        last_err = None
        bert = None
        for path, local_only in model_candidates:
            try:
                bert = BertModel.from_pretrained(path, local_files_only=local_only)
                break
            except Exception as e:
                last_err = e
                continue
        if bert is None:
            raise RuntimeError(
                f"Could not initialize BertModel. Tried: {[c[0] for c in model_candidates]}. "
                f"Last error: {repr(last_err)}"
            )
        self.bert = bert
        self.l0 = nn.Linear(768, 2)

    def forward(self, ids, mask, token_type_ids):
        out = self.bert(ids, attention_mask=mask, token_type_ids=token_type_ids)
        sequence_output = out.last_hidden_state
        logits = self.l0(sequence_output)
        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1)
        end_logits = end_logits.squeeze(-1)
        return start_logits, end_logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BERTBaseUncased().to(device)
model = nn.DataParallel(model)

weights_path = "/kaggle/input/tweet-sentiment-extraction-training/model.bin"
if os.path.exists(weights_path):
    state = torch.load(weights_path, map_location=device)
    model.load_state_dict(state)
else:
    print(
        f"WARNING: pretrained competition weights not found at {weights_path}. Using base bert weights."
    )

model.eval()




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 11.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.9 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 4.25.9 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 4.25.9 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.10.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
2026-05-11 07:51:41.336588: E external/local_xl

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

DataParallel(
  (module): BERTBaseUncased(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30522, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768, bias=True)

In [2]:
class TweetDataset(torch.utils.data.Dataset):
    def __init__(self, tweet, sentiment, selected_text):
        self.tweet = tweet
        self.sentiment = sentiment
        self.selected_text = selected_text
        self.max_len = config.MAX_LEN
        self.tokenizer = config.TOKENIZER

        if self.tokenizer is None:
            raise ValueError("config.TOKENIZER is not initialized (None).")

    def __len__(self):
        return len(self.tweet)

    def __getitem__(self, item):
        tweet = str(self.tweet[item])
        tweet = " ".join(tweet.split())

        selected_text = str(self.selected_text[item])
        selected_text = " ".join(selected_text.split())

        len_sel_text = len(selected_text)

        idx0 = -1
        idx1 = -1
        if len_sel_text > 0 and len(tweet) > 0:
            for ind in (i for i, e in enumerate(tweet) if e == selected_text[0]):
                if tweet[ind : ind + len_sel_text] == selected_text:
                    idx0 = ind
                    idx1 = ind + len_sel_text - 1
                    break

        char_targets = [0] * len(tweet)
        if idx0 != -1 and idx1 != -1:
            for j in range(idx0, idx1 + 1):
                if tweet[j] != " ":
                    char_targets[j] = 1

        enc = self.tokenizer(
            tweet,
            str(self.sentiment[item]),
            add_special_tokens=True,
            return_offsets_mapping=True,
            max_length=self.max_len,
            truncation=True,
            padding=False,
        )

        tok_tweet_ids = enc["input_ids"]
        tok_tweet_tokens = self.tokenizer.convert_ids_to_tokens(tok_tweet_ids)
        offsets = enc["offset_mapping"]

        try:
            sep_id = self.tokenizer.sep_token_id
            sep_positions = [i for i, tid in enumerate(tok_tweet_ids) if tid == sep_id]
            first_sep = (
                sep_positions[0] if len(sep_positions) > 0 else (len(tok_tweet_ids) - 1)
            )
        except Exception:
            first_sep = len(tok_tweet_ids) - 1

        tok_tweet_offsets = offsets[1:first_sep]

        targets = [0] * (len(tok_tweet_tokens) - 2)
        for j, (offset1, offset2) in enumerate(tok_tweet_offsets):
            if offset2 > offset1 and sum(char_targets[offset1:offset2]) > 0:
                if 1 + j < len(targets):
                    targets[1 + j] = 1
        if len(targets) < len(tok_tweet_tokens):
            targets = targets + [0] * (len(tok_tweet_tokens) - len(targets))
        targets = targets[: len(tok_tweet_tokens)]

        targets_start = [0] * len(targets)
        targets_end = [0] * len(targets)

        non_zero = np.nonzero(targets)[0]
        if len(non_zero) > 0:
            targets_start[non_zero[0]] = 1
            targets_end[non_zero[-1]] = 1

        mask = [1] * len(tok_tweet_ids)
        token_type_ids = enc.get("token_type_ids", [0] * len(tok_tweet_ids))

        padding_len = self.max_len - len(tok_tweet_ids)
        if padding_len < 0:
            tok_tweet_ids = tok_tweet_ids[: self.max_len]
            mask = mask[: self.max_len]
            token_type_ids = token_type_ids[: self.max_len]
            targets = targets[: self.max_len]
            targets_start = targets_start[: self.max_len]
            targets_end = targets_end[: self.max_len]
            offsets = offsets[: self.max_len]
            padding_len = 0
        else:
            tok_tweet_ids = tok_tweet_ids + [0] * padding_len
            mask = mask + [0] * padding_len
            token_type_ids = token_type_ids + [0] * padding_len
            targets = targets + [0] * padding_len
            targets_start = targets_start + [0] * padding_len
            targets_end = targets_end + [0] * padding_len
            offsets = offsets + [(0, 0)] * padding_len

        sentiment = [1, 0, 0]
        if self.sentiment[item] == "positive":
            sentiment = [0, 0, 1]
        if self.sentiment[item] == "negative":
            sentiment = [0, 1, 0]

        return {
            "ids": torch.tensor(tok_tweet_ids, dtype=torch.long),
            "mask": torch.tensor(mask, dtype=torch.long),
            "token_type_ids": torch.tensor(token_type_ids, dtype=torch.long),
            "targets": torch.tensor(targets, dtype=torch.long),
            "targets_start": torch.tensor(targets_start, dtype=torch.long),
            "targets_end": torch.tensor(targets_end, dtype=torch.long),
            "padding_len": torch.tensor(padding_len, dtype=torch.long),
            "offsets": torch.tensor(offsets, dtype=torch.long),
            "first_sep": torch.tensor(first_sep, dtype=torch.long),
            "orig_tweet": self.tweet[item],
            "orig_sentiment": self.sentiment[item],
            "orig_selected_text": self.selected_text[item],
        }




In [3]:
df_test = pd.read_csv("/kaggle/input/tweet-sentiment-extraction/test.csv")
df_test.loc[:, "selected_text"] = df_test.text.values

test_dataset = TweetDataset(
    tweet=df_test.text.values,
    sentiment=df_test.sentiment.values,
    selected_text=df_test.selected_text.values,
)

valid_data_loader = torch.utils.data.DataLoader(
    test_dataset, shuffle=False, batch_size=config.VALID_BATCH_SIZE, num_workers=0
)

fin_output_start = []
fin_output_end = []
fin_padding_lens = []
fin_offsets = []
fin_first_sep = []
fin_token_type_ids = []
fin_orig_sentiment = []
fin_orig_tweet = []

with torch.no_grad():
    for _, d in enumerate(valid_data_loader):
        ids = d["ids"].to(device, dtype=torch.long)
        token_type_ids = d["token_type_ids"].to(device, dtype=torch.long)
        mask = d["mask"].to(device, dtype=torch.long)

        token_type_ids_cpu = d["token_type_ids"].cpu().numpy()

        padding_len = d["padding_len"]
        offsets = d["offsets"].cpu().numpy()
        first_sep = d["first_sep"].cpu().numpy()

        orig_sentiment = d["orig_sentiment"]
        orig_tweet = d["orig_tweet"]

        o1, o2 = model(ids, mask, token_type_ids)

        fin_output_start.append(torch.sigmoid(o1).cpu().detach().numpy())
        fin_output_end.append(torch.sigmoid(o2).cpu().detach().numpy())
        fin_padding_lens.extend(padding_len.cpu().detach().numpy().tolist())

        fin_offsets.append(offsets)
        fin_first_sep.extend(first_sep.tolist())
        fin_token_type_ids.append(token_type_ids_cpu)
        fin_orig_sentiment.extend(list(orig_sentiment))
        fin_orig_tweet.extend(list(orig_tweet))

fin_output_start = np.vstack(fin_output_start)
fin_output_end = np.vstack(fin_output_end)
fin_offsets = np.vstack(fin_offsets)
fin_token_type_ids = np.vstack(fin_token_type_ids)

final_col = []

MAX_SPAN_LEN_TOKENS = 15
_EPS = 1e-9  # avoid log(0) while keeping probabilities essentially unchanged

for j in range(len(fin_orig_tweet)):
    padding_len = fin_padding_lens[j]
    original_tweet = str(fin_orig_tweet[j])
    sentiment = fin_orig_sentiment[j]

    if padding_len > 0:
        start_probs = fin_output_start[j, :][:-padding_len]
        end_probs = fin_output_end[j, :][:-padding_len]
        offsets = fin_offsets[j, :][:-padding_len]
        tt = fin_token_type_ids[j, :][:-padding_len]
        first_sep = int(fin_first_sep[j])
    else:
        start_probs = fin_output_start[j, :]
        end_probs = fin_output_end[j, :]
        offsets = fin_offsets[j, :]
        tt = fin_token_type_ids[j, :]
        first_sep = int(fin_first_sep[j])

    if sentiment == "neutral":
        final_col.append(str(original_tweet))
        continue

    # Change (score-improving, minimal): restrict decoding strictly to tweet region for pair inputs:
    # token_type_ids==0 AND before first [SEP] AND non-empty offsets (avoids [CLS]/[SEP]/sentiment part).
    valid_positions = np.where(
        (tt == 0) & (np.arange(len(tt)) < first_sep) & (offsets[:, 1] > offsets[:, 0])
    )[0]
    valid_positions = valid_positions[
        valid_positions != 0
    ]  # drop [CLS] if it slipped through

    if valid_positions.size == 0:
        final_col.append(str(original_tweet))
        continue

    s_probs = start_probs[valid_positions].astype(np.float32)
    e_probs = end_probs[valid_positions].astype(np.float32)

    # Change (score-improving, minimal): decode with log-prob sum instead of product for stability/length bias.
    s_scores = np.log(np.clip(s_probs, _EPS, 1.0))
    e_scores = np.log(np.clip(e_probs, _EPS, 1.0))

    L = len(valid_positions)
    joint = s_scores[:, None] + e_scores[None, :]  # (L,L)

    # enforce end >= start
    joint[np.tril_indices(L, k=-1)] = -1e9

    if MAX_SPAN_LEN_TOKENS is not None and MAX_SPAN_LEN_TOKENS >= 0:
        i_idx = np.arange(L)[:, None]
        j_idx = np.arange(L)[None, :]
        joint[(j_idx - i_idx) > MAX_SPAN_LEN_TOKENS] = -1e9

    best_flat = int(np.argmax(joint))
    best_i, best_k = divmod(best_flat, L)
    idx_start = int(valid_positions[best_i])
    idx_end = int(valid_positions[best_k])

    start_char = None
    end_char = None
    for t in range(idx_start, idx_end + 1):
        o1c, o2c = offsets[t]
        if o2c <= o1c:
            continue
        if start_char is None:
            start_char = int(o1c)
        end_char = int(o2c)

    if start_char is None or end_char is None or end_char <= start_char:
        final_output = ""
    else:
        final_output = original_tweet[start_char:end_char].strip()

    if len(str(original_tweet).split()) < 4:
        final_output = str(original_tweet)

    final_col.append(final_output)

submission = pd.read_csv(
    "/kaggle/input/tweet-sentiment-extraction/sample_submission.csv"
)
submission["selected_text"] = final_col

assert len(submission) == len(
    df_test
), f"Submission rows {len(submission)} != test rows {len(df_test)}"
assert list(submission.columns) == [
    "textID",
    "selected_text",
], f"Unexpected submission columns: {submission.columns.tolist()}"

submission.to_csv("submission.csv", index=False)
print(submission.head())
print("Wrote submission.csv with shape:", submission.shape)

       textID                                      selected_text
0  80a1e6bc32                                     saw a shooting
1  863097735d         i didnt get my tax returns! im so upset cu
2  264cd5277f  tired and didn`t really have an exciting Satur...
3  baee1e6ffc             i`ve been eating cheetos all morning..
4  67d06a8dee   haiiii sankQ i`m fineee ima js get a checkup ...
Wrote submission.csv with shape: (2749, 2)
